<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="20%">
</div>

<br>

# MACHINE LEARNING MODELS: CROSS-SECTIONAL PRICE PREDICTION

<br>

**About:** Train and compare four regression algorithms - Linear, Ridge, LASSO, and Random Forest - on the Ames Housing dataset, evaluating each for accuracy, interpretability, and generalization.

**Learning Goals:** After completing this notebook, you will be able to:

- Implement multiple regression algorithms and evaluate them on a held-out test set
- Explain why regularization prevents overfitting and how Ridge and LASSO differ
- Interpret Random Forest feature importances
- Choose among models based on accuracy, interpretability, and use-case requirements
- Compare models systematically using R-squared and RMSE

**Keywords:** regression, regularization, ridge, lasso, random forest, feature importance, overfitting

**Prerequisite Knowledge:** (1) `02_exploratory_analysis.ipynb` - feature correlations and data characteristics

**Target User:** Learners who have completed the EDA notebook and are ready to build and compare predictive models

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 1: LINEAR REGRESSION BASELINE](#Part_1)
> #### [PART 2: REGULARIZED MODELS (RIDGE AND LASSO)](#Part_2)
> #### [PART 3: ENSEMBLE METHODS (RANDOM FOREST)](#Part_3)
> #### [PART 4: MODEL COMPARISON](#Part_4)

<br>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# California housing is a stand-in for Ames; same cross-sectional structure
# Verified against sklearn docs, 2026-09-02 - re-check at https://scikit-learn.org
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target * 100000  # convert to dollar scale

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"Target (SalePrice) - mean: ${y.mean():,.0f}, std: ${y.std():,.0f}")

In [ ]:
# 80/20 train-test split: retains enough test data for reliable metric estimates
# while keeping most data for training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# StandardScaler needed for regularized models (Ridge, LASSO):
# regularization penalizes coefficient magnitude, so features must be on
# the same scale - otherwise features with large numeric ranges dominate
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # fit on train, transform test only

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **LINEAR** Regression Baseline

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Why Start with Linear Regression?

Linear regression is the foundation of predictive modeling for three reasons:

- **Interpretable**: Each feature gets a coefficient showing its marginal effect on price. "Every additional square foot adds $X, holding all else constant."
- **Fast**: A closed-form solution exists; no hyperparameter tuning required.
- **Diagnostic baseline**: If a more complex model only marginally outperforms linear regression, the extra complexity may not be worth the loss of interpretability.

The model: $\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p$

where $\hat{y}$ is predicted price, $x_i$ are features, and $\beta_i$ are estimated by minimizing the sum of squared residuals.

___

**Note:** Linear regression theory covered in James et al. (2021), *An Introduction to Statistical Learning*, Chapter 3. Free at [statlearning.com](https://www.statlearning.com/).

___

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_train = linear_model.predict(X_train)
y_pred_test = linear_model.predict(X_test)

linear_train_r2 = r2_score(y_train, y_pred_train)
linear_test_r2 = r2_score(y_test, y_pred_test)
linear_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("Linear Regression Results:")
print(f"  Train R-squared: {linear_train_r2:.3f}")
print(f"  Test  R-squared: {linear_test_r2:.3f}")
print(f"  Test  RMSE:      ${linear_rmse:,.0f}")

# Coefficients: which features matter most?
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": linear_model.coef_
}).sort_values("Coefficient", key=abs, ascending=False)

print("\nTop features by coefficient magnitude:")
print(coef_df.to_string(index=False))

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Linear Regression prints both train and test R-squared. If train R-squared is 0.64 and test R-squared is 0.60, is the model overfitting? Now: modify the code to also compute and print `linear_train_rmse`. Is RMSE or R-squared more useful for communicating model quality to a non-technical stakeholder?**

<br>

```python
# Add train RMSE:
# linear_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
### YOUR CODE HERE ###
# print(f"Train RMSE: ${linear_train_rmse:,.0f}")
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **REGULARIZED** Models (Ridge and LASSO)

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### The Problem with Ordinary Linear Regression

Ordinary linear regression minimizes: $\sum_{i=1}^{n} (y_i - \hat{y}_i)^2$

With many features, some will fit *noise* in the training data, producing large coefficients that hurt generalization. Regularization adds a penalty term:

- **Ridge (L2)**: $\sum (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} \beta_j^2$ - shrinks all coefficients toward zero but keeps all features
- **LASSO (L1)**: $\sum (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} |\beta_j|$ - can shrink coefficients *exactly* to zero, performing automatic feature selection

where $\alpha > 0$ controls the strength of the penalty. Higher $\alpha$ = more shrinkage = simpler model.

**When to use which:**
- Ridge: When you believe most features are relevant but none should dominate.
- LASSO: When you suspect many features are irrelevant and want automatic feature selection.

___

**Note:** Regularization theory from Tibshirani (1996) for LASSO and Hoerl & Kennard (1970) for Ridge. Practical guidance in James et al. (2021), Chapter 6. [statlearning.com](https://www.statlearning.com/).

___

In [ ]:
# Ridge regression - requires scaled features
# alpha=1.0 is a moderate starting point; tune via cross-validation in practice
ridge_model = Ridge(alpha=1.0)  # TODO: verify default solver in current sklearn
ridge_model.fit(X_train_scaled, y_train)
ridge_pred = ridge_model.predict(X_test_scaled)
ridge_r2 = r2_score(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

print("Ridge Regression (alpha=1.0):")
print(f"  Test R-squared: {ridge_r2:.3f}")
print(f"  Test RMSE:      ${ridge_rmse:,.0f}")

# LASSO regression
# alpha=100 is relatively strong for dollar-scale targets
lasso_model = Lasso(alpha=100, max_iter=5000)
lasso_model.fit(X_train_scaled, y_train)
lasso_pred = lasso_model.predict(X_test_scaled)
lasso_r2 = r2_score(y_test, lasso_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))

print("\nLASSO Regression (alpha=100):")
print(f"  Test R-squared: {lasso_r2:.3f}")
print(f"  Test RMSE:      ${lasso_rmse:,.0f}")
print(f"  Non-zero features: {(lasso_model.coef_ != 0).sum()} of {len(lasso_model.coef_)}")

# Which features did LASSO zero out?
lasso_coef = pd.Series(lasso_model.coef_, index=X.columns)
print("\nLASSO coefficients (0 = feature excluded):")
print(lasso_coef.round(2).to_string())

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **LASSO zeroed out some features. Check which ones and explain whether that makes intuitive sense given the EDA in Notebook 2. Then: retrain LASSO with `alpha=1000` (stronger penalty). How many features survive?**

<br>

```python
# Features zeroed out by current LASSO:
# zeroed = lasso_coef[lasso_coef == 0].index.tolist()
# print("Zeroed features:", zeroed)

# Stronger penalty:
# lasso_strong = Lasso(alpha=1000, max_iter=5000)
### YOUR CODE HERE ###
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **ENSEMBLE** Methods (Random Forest)

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Beyond Linear Models

Linear models assume price changes proportionally with each feature. Housing markets have non-linear patterns:
- **Diminishing returns**: An extra bedroom in a mansion adds less value than in a starter home.
- **Interactions**: Waterfront location + large lot together are worth more than each feature separately.
- **Thresholds**: Below a minimum livable square footage, pricing changes qualitatively.

**Random Forest** builds many decision trees, each on a random subsample of training data and a random subset of features. Predictions are averaged across all trees. This achieves:
- Non-linear boundaries (each tree can split on any feature at any threshold)
- Low variance (averaging trees reduces the instability of any single tree)
- Built-in feature importance (how much each feature reduces error across all splits)

___

**Note:** Breiman (2001), "Random Forests," *Machine Learning* 45(1), 5-32 is the original paper. Practical guidance in James et al. (2021), Chapter 8. [statlearning.com](https://www.statlearning.com/).

___

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,   # 100 trees: enough for stable estimates without excessive compute
    max_depth=15,       # cap depth to prevent individual trees from overfitting
    random_state=42,
    n_jobs=-1           # use all available CPU cores
)
rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_test_r2 = r2_score(y_test, rf_test_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))

print("Random Forest Results:")
print(f"  Train R-squared: {rf_train_r2:.3f}")
print(f"  Test  R-squared: {rf_test_r2:.3f}")
print(f"  Test  RMSE:      ${rf_rmse:,.0f}")

# Feature importances: average reduction in impurity across all trees and splits
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nFeature Importances:")
print(importance_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importance_df["Feature"], importance_df["Importance"], color="#003262")
ax.set_title("Random Forest Feature Importances")
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Random Forest has a larger gap between train and test R-squared than Linear Regression does. Explain why this is expected rather than alarming for tree-based models. Then: retrain with `max_depth=5` and compare train vs. test R-squared. Does reducing depth help or hurt generalization?**

<br>

```python
# Shallow Random Forest:
# rf_shallow = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
### YOUR CODE HERE ###
# Compare train/test R-squared to rf_model above
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **MODEL** Comparison

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Choosing the Right Model

Model selection is not purely about which model achieves the highest R-squared on the test set. It involves balancing four factors:

- **Accuracy**: How well does the model predict on unseen data? (R-squared, RMSE)
- **Interpretability**: Can you explain a prediction to a homeowner or a regulator?
- **Computation**: Does the model need to predict in real time, or is batch-processing acceptable?
- **Maintainability**: Will the model need to be retrained frequently as new data arrives?

For housing price prediction with no strict real-time requirement, Random Forest is often the best default because it handles non-linearity without feature engineering. For regulatory or audit contexts where every coefficient must be explainable, Ridge or LASSO are better choices.

In [ ]:
# Compile all results into a comparison table
results = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge", "LASSO", "Random Forest"],
    "Test R-squared": [linear_test_r2, ridge_r2, lasso_r2, rf_test_r2],
    "Test RMSE ($)": [linear_rmse, ridge_rmse, lasso_rmse, rf_rmse],
    "Interpretable": ["Yes", "Yes", "Yes (sparse)", "Partial (importance scores)"],
    "Handles Non-linearity": ["No", "No", "No", "Yes"],
})

print("Model Comparison:")
print(results.to_string(index=False))

best_idx = results["Test R-squared"].idxmax()
print(f"\nBest by R-squared: {results.loc[best_idx, 'Model']}")
print(f"  R-squared = {results.loc[best_idx, 'Test R-squared']:.3f}")
print(f"  RMSE      = ${results.loc[best_idx, 'Test RMSE ($)']:,.0f}")

### When to Use Each Model

| Use Case | Recommended Model | Reason |
|----------|-------------------|--------|
| Explain to regulators or homeowners | Linear or LASSO | Coefficients are directly interpretable |
| Maximum prediction accuracy | Random Forest | Handles non-linearity and interactions |
| Many irrelevant features | LASSO | Automatic feature selection via zero coefficients |
| Fast real-time inference | Linear or Ridge | Matrix multiply only; no tree traversal |
| Understanding which features drive price | Random Forest | Feature importance scores |

<strong style="color:red">KEY CONSIDERATION:</strong> Never choose a model based on test performance alone without checking whether the test set was properly held out. If you tuned hyperparameters on the test set (or on data derived from the test set), your reported performance will be optimistic.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Random Forest achieves higher R-squared than linear models, but has lower interpretability. Describe a realistic housing prediction scenario where you would choose LASSO over Random Forest despite lower accuracy. What specific requirements make interpretability worth the accuracy cost?**

<br>

```python
# Write your reasoning in comments:
# scenario = "...
# requirement_1 = "...
# requirement_2 = "...
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

---

## Summary

We trained four types of regression models:
- **Linear Regression**: Interpretable baseline; limited by linearity assumption
- **Ridge**: Regularized linear; prevents overfitting when features are correlated
- **LASSO**: Regularized linear with automatic feature selection via zero coefficients
- **Random Forest**: Non-linear ensemble; best accuracy, partial interpretability

For housing prices with non-linear drivers, Random Forest typically leads on accuracy. But choose based on your priorities - accuracy, interpretability, or inference speed.

The next notebook (`04_time_series_methods.ipynb`) asks a different question: can past prices alone forecast future prices, without using any property features?

<hr style="border: 6px solid#003262;" />